# M5.S3 · Containers, portability and reproducibility in HPC
IE University · Bachelor in Computer Science and Artificial Intelligence · 2026–27 · Official Session 23

Why does a working scientific calculation fail, or change its answer, on another computer?

**Learning outcomes:** diagnose dependency and process-environment differences; distinguish modules, package specifications and containers; record and replay a numerical experiment; explain host/container boundaries; invalidate stale workflow results.

**Five experiments:** E1 dependency drift → E2 process environments → E3 recorded temperature analysis and C/OpenMP → E4 the same payload in Apptainer → E5 workflow cache and independent handover.

Each experiment follows **Real problem → Prediction → Code → Run → Result → Explanation → Student experiment → Takeaway**. Core Python examples are real executions with tiny inputs. Slurm, C/OpenMP and Apptainer results require their own evidence.

**80-minute class:** opening/setup 6 min; E1 8; E2 13; E3 17; E4 17; E5 13; recap/exit 6. Optional reference material is outside this time budget.


## 0 · Start with SSH, then use Jupyter

1. **First**, connect to the SciTech cluster and complete the software **modules** practical below. Inspect GCC, MPI and HDF5, compile a tiny C program, and restore your module environment.
2. **Next**, complete the PyTorch **Apptainer container** practical using a separate folder in your home directory. Run the seeded training experiment twice and change one parameter.
3. **Finally**, open https://jupyter.iesci.tech:8085/ and save a personal copy of this notebook in your student-owned directory shared with compute nodes. Run the Jupyter cells in order.

The notebook's main Python exercises do not install packages. Its existing Slurm CPU job tests native C/OpenMP and the container examples when available (`RUN_CLUSTER=True`, `RUN_CONTAINER=True`). It requests one node, one task, two CPUs, 1 GiB, and three minutes. The Slurm container selection can use `IMAGE=""` to discover suitable shared images, or a readable instructor-approved SIF path.

No image pull, software installation, GPU request or recursive filesystem search is required. The notebook labels PASS, FAIL, SKIPPED and NOT RUN separately. A Slurm job can remain QUEUED/RUNNING before collection; follow the collection instructions without resubmitting.


In [ ]:
from pathlib import Path
import os, sys, json, math, random, hashlib, platform, shutil, subprocess, csv, time, venv, uuid, zipfile
ROOT = (Path.cwd() / "m5s3_repro_lab").resolve()
ROOT.mkdir(parents=True, exist_ok=True)
RUN_CLUSTER = True
RUN_CONTAINER = True
IMAGE = os.environ.get("M5S3_IMAGE", "")  # leave empty to discover an existing course image
IMAGE_DIRS = ["/data/software/containers", str(Path.home()/"projects/def-sponsor00/shared/images")]
# Add another instructor-approved directory here if the site stores images elsewhere.
EXPECTED_IMAGE_SHA256 = ""  # optional: pin the hash reported by a previous accepted run
LOAD_SCALE = 1.0            # NumPy example: change just this force multiplier (0.1 to 10)
MODULE = "foss/2023b"  # established SciTech toolchain; inspect actual module status in job log
THRESHOLD_C = 25.0     # one-parameter experiment: 20.0 or 22.0
NEW_RUN = False       # True creates a fresh attempt; do not use while your job is queued/running
EVIDENCE = {}
def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()
def call(args, **kw):
    r = subprocess.run([str(x) for x in args], capture_output=True, text=True, timeout=30, **kw)
    if r.returncode: raise RuntimeError(r.stderr or r.stdout or str(r.returncode))
    return r.stdout.strip()
print("LOCAL NOTEBOOK", platform.node(), sys.version.split()[0], sys.executable)
print("Work directory:", ROOT)
print("Slurm submission available:", bool(shutil.which("sbatch")))
print("C/container cases: NOT RUN until a completed batch report is collected.")


## First practical · Explore HPC software modules by SSH

Start with **modules**, then continue to the PyTorch container exercise below. Modules select software already installed on the cluster; they do not install packages into your account. Allow about **10–15 minutes**.

### 1 · Connect and create your folder

From **PowerShell** (Windows) or **Terminal** (macOS/Linux), use your own account:

```bash
ssh YOUR_USERNAME@ssh.iesci.tech
```

The following commands run **after SSH login**, on the cluster. You should follow the cluster rules for compute-node use. This short software inspection and tiny compilation needs no GPU.

```bash
mkdir -p "$HOME/hpc_course/m5s3_modules_lab"
cd "$HOME/hpc_course/m5s3_modules_lab"
pwd
hostname
```

### 2 · Inspect what is installed

```bash
module list
module avail
module spider HDF5
module spider OpenMPI
command -v gcc
gcc --version | head -1
command -v python3
python3 --version
```

`module list` shows which optional modules are currently loaded, not every program available on the system. On our cluster, default Python may already come from **EESSI/CVMFS** even when `module list` reports no modules loaded.

**Checkpoint 1:** write down the initial GCC version and the path to Python. Find at least one HDF5 and OpenMPI version advertised by `module spider`.

### 3 · Load a scientific software stack

We use the EESSI 2023b compiler/MPI toolchain already visible on the course system:

```bash
module load foss/2023b
module list
command -v gcc
gcc --version | head -1
command -v mpicc
mpicc --version | head -1
```

If the toolchain is not visible, check `module spider foss`; do not install software. This is a software selection exercise, not a request to modify the cluster.

### 4 · Compile and execute a small C program

Create a program that prints the compiler version and calculates the sum of the integers from 1 to 100,000.

```bash
cat > module_demo.c <<'C'
#include <stdio.h>

int main(void) {
    long long total = 0;
    for (int i = 1; i <= 100000; ++i) total += i;
    printf("Compiler: %s\\n", __VERSION__);
    printf("Sum 1..100000: %lld\\n", total);
    return 0;
}
C

gcc -O2 -Wall module_demo.c -o module_demo
./module_demo | tee module_result.txt
```

**Checkpoint 2:** the sum must be `5000050000`. The compiler line is recorded from the actual compiler used. The compilation happened on the cluster with software selected through modules.

### 5 · Explore a scientific library without installing it

```bash
module spider HDF5/1.14.3-gompi-2023b
module load HDF5/1.14.3-gompi-2023b
module list
command -v h5cc
command -v h5pcc
```

Depending on the HDF5 module, one or both compiler wrappers may be available. The important result is that the HDF5 installation is accessible without `pip`, `apt`, or compiling HDF5 yourself.

### 6 · Return to the initial environment

```bash
module purge
module list
command -v gcc
command -v python3
cat module_result.txt
```

The `module_demo` executable and `module_result.txt` stay in your **home folder** after unloading the modules. However, executables built with particular shared libraries may still require their original module environment when run.

**Checkpoint 3:** which software paths changed, and which files remained? Explain the difference between **selecting preinstalled software** and **packaging software into a container**.

Keep the results in `~/hpc_course/m5s3_modules_lab`. Now continue to the PyTorch Apptainer lab, where the Python runtime and PyTorch library come from a shared container image rather than your selected host modules.


## First experiment · Connect by SSH and run PyTorch containers

We will begin with a practical exercise on the SciTech cluster. Rather than just adding three numbers, you will **train a small machine-learning model using PyTorch inside an Apptainer container**, run a second container process to check reproducibility, then change the experiment and compare the result. Allow around 15–20 minutes to explore and answer the questions.

Our model learns a straight line from 256 artificial observations. The underlying relationship is approximately `y = 3x + 2`. You should see the learned slope approach 3 and the intercept approach 2. This is **CPU-only**; do not request a GPU or install packages.

### 1 · Connect to the cluster

On your computer, open PowerShell or Terminal. Replace the placeholder with **your own** SciTech username:

```bash
ssh YOUR_USERNAME@ssh.iesci.tech
```

The remaining commands are run in the cluster SSH terminal. Follow the course's compute-node policy; run this short CPU exercise on an authorized node or allocation, not as a sustained workload on a busy login node.

### 2 · Work in your own folder

```bash
mkdir -p "$HOME/hpc_course/m5s3_container_lab"
cd "$HOME/hpc_course/m5s3_container_lab"
pwd
hostname
```

Keep your scripts and results here, not in `/tmp`.

### 3 · Inspect the shared container

```bash
IMAGE=/data/software/containers/pytorch-2.8.0-cuda12.8.sif
command -v apptainer
ls -lh "$IMAGE"
apptainer exec --cleanenv "$IMAGE" /opt/conda/bin/python3 -c 'import torch; print("PyTorch:", torch.__version__)'
```

The SIF image is already installed at the cluster. Do not copy, download or rebuild it. The `/opt/conda/bin/python3` interpreter is the expected one for this image. If a check fails, save the error and ask for assistance rather than installing libraries into the host Python environment.

### 4 · Create a training program

Copy this entire block into your SSH terminal. It creates `train_model.py` in the working folder.

```bash
cat > train_model.py <<'PY'
import sys, json, hashlib
import torch

seed = int(sys.argv[1])
true_slope = float(sys.argv[2])
torch.set_num_threads(2)
torch.manual_seed(seed)

x = torch.linspace(-2, 2, 256).reshape(-1, 1)
y = true_slope * x + 2.0 + 0.05 * torch.sin(x * 9)
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(240):
    optimizer.zero_grad()
    loss = torch.mean((model(x) - y) ** 2)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    learned_slope = float(model.weight.item())
    learned_intercept = float(model.bias.item())
    error = float(torch.mean(torch.abs(model(x) - y)).item())

result = {
    "torch": torch.__version__,
    "seed": seed,
    "target_slope": true_slope,
    "learned_slope": round(learned_slope, 6),
    "learned_intercept": round(learned_intercept, 6),
    "mean_absolute_error": round(error, 6),
}
result["fingerprint"] = hashlib.sha256(
    json.dumps(result, sort_keys=True).encode()
).hexdigest()[:12]
print("M5S3_TRAIN=" + json.dumps(result, sort_keys=True))
PY
```

What the code does: it generates observations, creates a one-layer PyTorch model, trains it using gradient descent, and prints a result record with the learned coefficients, error and a short fingerprint.

### 5 · Launch two separate container executions

These are **two separate container processes**, not two virtual machines. We pass the same script and the same parameters to both.

```bash
apptainer exec --cleanenv "$IMAGE" /opt/conda/bin/python3 - 42 3.0 < train_model.py | tee run_A.txt
apptainer exec --cleanenv "$IMAGE" /opt/conda/bin/python3 - 42 3.0 < train_model.py | tee run_B.txt
diff run_A.txt run_B.txt && echo "CHECKPOINT 1: reproducible"
```

Using `< train_model.py` sends the script through standard input. **You do not need bind mounts or a `/work` folder inside the container.** The output files are created by the host shell inside your own folder.

Look at `learned_slope`, `learned_intercept`, `mean_absolute_error` and `fingerprint`. The two outputs should be identical because the random seed, data, parameters, algorithm and image are the same.

### 6 · Change an input parameter

The third run uses the same container and program but changes the underlying slope from 3 to 4.

```bash
apptainer exec --cleanenv "$IMAGE" /opt/conda/bin/python3 - 42 4.0 < train_model.py | tee run_C.txt
diff run_A.txt run_C.txt || echo "CHECKPOINT 2: changed parameters produce different results"
```

Now the learned slope should approach **4**, while the intercept stays close to **2**. The output and fingerprint should change.

### 7 · Explore the environment

```bash
apptainer exec --cleanenv "$IMAGE" /opt/conda/bin/python3 -c 'import sys,torch; print("Python:",sys.version.split()[0]); print("PyTorch:",torch.__version__); print("CPU threads:",torch.get_num_threads())'
ls -lh run_*.txt train_model.py
```

You can repeat a run using seed `7` instead of `42` and examine how the learned parameters vary. Do not change the installed software or image.

### Check your understanding

- Why do runs A and B produce the same output even though Apptainer starts two separate processes?
- Why does C change when we have not changed the code or container?
- What is inside the image, and which files remain in your home folder?
- Would repeating this with a different PyTorch image necessarily produce identical floating-point results?

**Checkpoint 1** is two identical outputs. **Checkpoint 2** is a changed result after changing an experiment parameter. You should have `train_model.py`, `run_A.txt`, `run_B.txt` and `run_C.txt` in `~/hpc_course/m5s3_container_lab`.

At the end of this notebook we perform the **same three runs automatically** from Jupyter and display their results. You do not have to copy your SSH files to Jupyter.


## E1 · The same application, a changed dependency
**Real problem:** a temperature conversion changes after a dependency update. We create two tiny local teaching versions of `thermo`; this is not a claim about a published package.

**Prediction:** will identical application text always give the same Kelvin value? Which part of the environment might explain a difference?

**Code / Run:** execute fresh Python processes in the two dependency folders.


In [ ]:
TEMPERATURE_C = 20.0
app = f"import thermo; print(round(thermo.to_kelvin({TEMPERATURE_C}), 3))"
versions = {"v1": "def to_kelvin(x): return x + 273.15\n",
            "v2": "def to_kelvin(x): return (x - 32) * 5 / 9 + 273.15\n"}
observed = {}
for name, source in versions.items():
    folder = ROOT / name
    folder.mkdir(exist_ok=True)
    (folder / "thermo.py").write_text(source)
    observed[name] = call([sys.executable, "-c", app], cwd=folder)
    print(name, "->", observed[name], "K")
assert float(observed["v1"]) == round(TEMPERATURE_C+273.15,3) and observed["v1"] != observed["v2"]
EVIDENCE["E1_dependency_drift"] = "PASS"
print("LOCAL PASS: identical app text, different dependency behaviour.")


**Result / Explanation:** inspect the two actual values. Version 2 uses Fahrenheit semantics. The program can run successfully and still answer the wrong scientific question.

**Student experiment:** change only `TEMPERATURE_C` to 0; predict each answer and rerun. Keep the dependency versions unchanged.

**Takeaway:** record code **and** dependency identity, units and API assumptions.


### Checkpoint Q1
The script and input are unchanged, but a dependency update changes the result. Which evidence helps most?

- **A.** Only the final screenshot
- **B.** Script, input identity and exact dependency environment
- **C.** Only elapsed time
- **D.** Only the input filename


**Checkpoint Q1 · Answer: B**

The script, input identity and exact dependency environment show what changed. A screenshot loses provenance; timing alone does not identify dependencies; a filename does not verify file contents or software.


## E2 · Which environment does this process use?
**Real problem:** loading a module in a terminal does not switch an already-running Jupyter kernel.

**Prediction:** can a child process change the parent’s environment? Does a virtual environment include an independent Linux kernel?

**Code / Run:** inspect a real child environment and create an empty real venv without pip or downloads.


In [ ]:
child = "import os; print(os.environ.get('M5S3_SETTING', 'unset'))"
clean = dict(os.environ); clean.pop("M5S3_SETTING", None)
selected = dict(clean, M5S3_SETTING="two-threads")
assert call([sys.executable, "-c", child], env=clean) == "unset"
assert call([sys.executable, "-c", child], env=selected) == "two-threads"
assert os.environ.get("M5S3_SETTING") == clean.get("M5S3_SETTING")
print("PASS: child sees its supplied environment; parent is unchanged.")
EVIDENCE["E2_child_environment"] = "PASS"
env_dir = ROOT / "empty_venv"
try:
    venv.EnvBuilder(with_pip=False).create(env_dir)
    vpy = env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    result = json.loads(call([vpy, "-c", "import sys,json; print(json.dumps({'is_venv':sys.prefix!=sys.base_prefix,'python':sys.version.split()[0]}))"]))
    assert result["is_venv"]
    print("PASS: real venv:", result, "(no packages installed)")
    EVIDENCE["E2_venv"] = "PASS"
except (OSError, RuntimeError) as exc:
    print("SKIPPED: venv creation unavailable:", exc)
    EVIDENCE["E2_venv"] = "SKIPPED"


In [ ]:
# Read-only inspection: module is often a shell function, so use bash -lc.
probe = subprocess.run(["bash", "-lc", "if command -v module >/dev/null 2>&1; then module list 2>&1; else echo 'SKIPPED: module command unavailable in this shell'; fi"],
                       capture_output=True, text=True, timeout=15)
print(probe.stdout or probe.stderr)
print("Current kernel still:", sys.executable)
print("This probes the current host; the compute job records its own module status.")


**Result / Explanation:** a venv separates Python package locations while reusing a base interpreter and host system. A module selects site-installed software in a shell. It does not install it into the notebook.

**Spack concept:** `spack.yaml` states requirements; concretization resolves a dependency graph into `spack.lock`. We write an example below; **Spack concretization and installation are NOT RUN**. Preserve the lockfile, Spack version and site/compiler configuration for a real release. A lockfile does not promise universal binary compatibility.

**Student experiment:** change only `M5S3_SETTING` in `selected` to `four-threads`, rerun that child call, and verify the parent stays unchanged.

**Takeaway:** identify the interpreter and environment of the process that actually executes the code.


In [ ]:
spack_yaml = "spack:\n  specs:\n    - zlib\n  concretizer:\n    unify: true\n"
(ROOT / "spack.yaml").write_text(spack_yaml)
print(spack_yaml)
print("NOT RUN: spack concretize / spack install. No lockfile or installation is invented.")


### Checkpoint Q2
A student loads a Python module in a terminal, then returns to an existing notebook kernel. What should they expect?

- **A.** The kernel switches interpreter immediately
- **B.** The module builds a new container
- **C.** The terminal changes; the existing kernel keeps its interpreter
- **D.** The input file changes automatically


**Checkpoint Q2 · Answer: C**

Changes made in a terminal do not change an already running Jupyter kernel. The existing kernel continues to use its current interpreter; loading a module does not build a container or change the input data.


## E3 · Record one scientific calculation
**Real problem:** compute the mean temperature and fraction above a threshold from the same 1,000 sensor readings.

**Prediction:** if only the threshold changes, should the input hash or mean change? Should the fraction change?

**Code / Run:** reuse the standard-library `analysis.py` payload from the earlier M5S3 notebook. The source cell also embeds a small C/OpenMP implementation and the batch/replay helpers so no repository checkout is required. The cell only writes sources; it does not run C on the login node. Expand it to inspect the full code.


In [ ]:
SOURCES = {'analysis.py': 'import argparse, csv, json, hashlib, math, platform, sys, os\nfrom pathlib import Path\np = argparse.ArgumentParser()\np.add_argument("--input", required=True)\np.add_argument("--threshold", type=float, default=25.0)\np.add_argument("--output", required=True)\na = p.parse_args()\ninput_path = Path(a.input)\nwith input_path.open(newline="", encoding="utf-8") as f:\n    values = [float(row["temperature_c"]) for row in csv.DictReader(f)]\nif not values:\n    raise ValueError("input must contain temperatures")\nresult = {"count": len(values), "mean_c": math.fsum(values) / len(values),\n          "fraction_above": sum(x > a.threshold for x in values) / len(values)}\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\nrecord = {"result": result, "parameters": {"threshold_c": a.threshold},\n          "input_sha256": sha(input_path), "code_sha256": sha(__file__),\n          "runtime": {"python": sys.version.split()[0],\n                      "platform": platform.system(), "machine": platform.machine(),\n                      "executable": sys.executable,\n                      "omp_threads": os.environ.get("OMP_NUM_THREADS", "unset")}}\nPath(a.output).write_text(json.dumps(record, indent=2) + "\\n", encoding="utf-8")\nprint(json.dumps(result, sort_keys=True))\n', 'temperature.c': '#include <stdio.h>\n#include <stdlib.h>\n#include <math.h>\n#ifdef _OPENMP\n#include <omp.h>\n#endif\nint main(int argc, char **argv) {\n    if (argc != 4) return 2;\n    double threshold = atof(argv[2]), values[2000];\n    int threads = atoi(argv[3]), n = 0, seen = 1;\n    if (threads < 1 || threads > 2) return 2;\n    FILE *f = fopen(argv[1], "r");\n    char line[256];\n    if (!f || !fgets(line, sizeof line, f)) return 3;\n    while (fgets(line, sizeof line, f)) {\n        double x;\n        if (n >= 2000 || sscanf(line, "%lf", &x) != 1 || !isfinite(x)) return 4;\n        values[n++] = x;\n    }\n    fclose(f);\n    if (!n) return 5;\n    double sum = 0;\n    int above = 0;\n#ifdef _OPENMP\n    omp_set_dynamic(0);\n    #pragma omp parallel num_threads(threads) reduction(+:sum,above)\n#endif\n    {\n#ifdef _OPENMP\n        #pragma omp single\n        seen = omp_get_num_threads();\n        #pragma omp for schedule(static)\n#endif\n        for (int i = 0; i < n; i++) {\n            sum += values[i];\n            above += values[i] > threshold;\n        }\n    }\n    if (!isfinite(sum)) return 6;\n    printf("{\\"count\\":%d,\\"mean_c\\":%.17g,\\"fraction_above\\":%.17g,\\"threads\\":%d}\\n",\n           n, sum/n, (double)above/n, seen);\n    return 0;\n}\n', 'worker.py': 'import hashlib, json, math, os, platform, shutil, subprocess, sys\nfrom pathlib import Path\nROOT = Path(__file__).resolve().parent\nos.chdir(ROOT)\ncfg = json.loads((ROOT / "settings.json").read_text())\nreport = {"run_id":cfg["run_id"], "context":"SCITECH_SLURM" if os.environ.get("SLURM_JOB_ID") else "LOCAL_AUTHOR_VALIDATION",\n          "job_id":os.environ.get("SLURM_JOB_ID"), "host":platform.node(), "python":sys.version,\n          "module_status":os.environ.get("M5S3_MODULE_STATUS", "not probed locally"),\n          "cases":{}}\ndef sha(p):\n    h = hashlib.sha256()\n    with Path(p).open("rb") as f:\n        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)\n    return h.hexdigest()\ndef call(args, **kw):\n    r = subprocess.run([str(x) for x in args], capture_output=True, text=True, timeout=25, **kw)\n    if r.returncode: raise RuntimeError(" ".join(map(str,args))+"\\n"+r.stderr[-3000:]+r.stdout[-1000:])\n    return r.stdout.strip()\ndef agree(a,b):\n    return a["count"] == b["count"] and all(math.isfinite(a[k]) and math.isclose(a[k],b[k],rel_tol=1e-12,abs_tol=1e-12) for k in ("mean_c","fraction_above"))\ndef record(name, status, **kw):\n    report["cases"][name] = dict(status=status, **kw)\n    print(name, json.dumps(report["cases"][name], sort_keys=True), flush=True)\ndef run():\n    for name,h in cfg["source_sha256"].items():\n        if sha(ROOT/name) != h: raise RuntimeError("Frozen input/source changed: "+name)\n    call([sys.executable,"analysis.py","--input","temperatures.csv","--threshold",cfg["threshold_c"],"--output","native.json"])\n    ref=json.loads((ROOT/"native.json").read_text())["result"]\n    expected=cfg["reference"]\n    if not agree(ref,expected): raise RuntimeError("Native Python reference mismatch")\n    record("python", "PASS", result=ref)\n    compiler=shutil.which("gcc")\n    if not compiler:\n        record("c_serial","SKIPPED",reason="gcc unavailable in job")\n        record("c_openmp","SKIPPED",reason="gcc unavailable in job")\n    else:\n        report["compiler_path"]=compiler\n        report["compiler_version"]=call([compiler,"--version"]).splitlines()[0]\n        try:\n            call([compiler,"-std=c11","-O2","-Wall","-Wextra","temperature.c","-lm","-o","temperature_serial"])\n            actual=json.loads(call([ROOT/"temperature_serial","temperatures.csv",cfg["threshold_c"],1]))\n            record("c_serial","PASS" if agree(actual,ref) else "FAIL",result=actual)\n        except Exception as e: record("c_serial","FAIL",reason=str(e))\n        # Probe actual OpenMP compilation separately from scientific correctness.\n        probe=ROOT/"omp_probe.c"\n        probe.write_text("#include <omp.h>\\nint main(void){return omp_get_max_threads()<1;}\\n")\n        r=subprocess.run([compiler,"-fopenmp",str(probe),"-o","omp_probe"],capture_output=True,text=True,timeout=25)\n        if r.returncode:\n            record("c_openmp","SKIPPED",reason="OpenMP compiler/runtime unavailable",diagnostic=r.stderr[-2000:])\n        else:\n            try:\n                call([compiler,"-std=c11","-O2","-Wall","-Wextra","-fopenmp","temperature.c","-lm","-o","temperature_omp"])\n                env=dict(os.environ,OMP_NUM_THREADS="2",OMP_DYNAMIC="FALSE",OMP_PROC_BIND="close",OMP_PLACES="cores")\n                actual=json.loads(call([ROOT/"temperature_omp","temperatures.csv",cfg["threshold_c"],2],env=env))\n                record("c_openmp","PASS" if agree(actual,ref) and actual["threads"]==2 else "FAIL",result=actual)\n            except Exception as e: record("c_openmp","FAIL",reason=str(e))\n    from container_examples import run_examples\n    run_examples(cfg, ROOT, report, record, sha, agree)\n\ntry:\n    run()\nexcept Exception as e:\n    record("worker","FAIL",reason=repr(e))\nreport["complete"]=True\nreport["status"]="FAIL" if any(x["status"]=="FAIL" for x in report["cases"].values()) else "PASS"\nreport["note"]="Job PASS does not promote SKIPPED capabilities to PASS; inspect each case. No MPI/GPU experiment is performed."\n# Atomic publication: a partial write is never treated as a completed report.\n(ROOT/"report.json.tmp").write_text(json.dumps(report,indent=2)+"\\n")\n(ROOT/"report.json.tmp").replace(ROOT/"report.json")\nprint("M5S3_JOB_DONE status="+report["status"],flush=True)\nraise SystemExit(1 if report["status"]=="FAIL" else 0)\n', 'replay.py': 'import hashlib, json, math, subprocess, sys\nfrom pathlib import Path\nroot = Path(__file__).resolve().parent\nmanifest = json.loads((root / "manifest.json").read_text())\nfor name, expected in manifest["files"].items():\n    actual = hashlib.sha256((root / name).read_bytes()).hexdigest()\n    if actual != expected:\n        print("FAIL: integrity mismatch:", name)\n        raise SystemExit(2)\nresult = subprocess.run([sys.executable, "analysis.py", "--input", "temperatures.csv",\n    "--threshold", str(manifest["threshold_c"]), "--output", "replayed.json"],\n    cwd=root, capture_output=True, text=True, timeout=20)\nif result.returncode:\n    print("FAIL:", result.stderr)\n    raise SystemExit(3)\na = json.loads((root / "replayed.json").read_text())["result"]\nb = json.loads((root / "expected.json").read_text())["result"]\nok = a["count"] == b["count"] and all(math.isclose(a[k], b[k], rel_tol=1e-12, abs_tol=1e-12)\n    for k in ("mean_c", "fraction_above"))\nprint("PASS: independent replay agrees" if ok else "FAIL: numerical mismatch")\nraise SystemExit(0 if ok else 4)\n', 'experiment.def': 'Bootstrap: docker\nFrom: python:3.11-slim\n\n%labels\n    Course HPC_M5S3\n\n%test\n    python3 -c "import csv, json, math"\n\n%runscript\n    exec python3 "$@"\n', 'spack.yaml': 'spack:\n  specs:\n    - zlib\n  concretizer:\n    unify: true\n', 'run.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s3-repro\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=2\n#SBATCH --mem=1G\n#SBATCH --time=00:03:00\n#SBATCH --output=job-%j.out\n#SBATCH --error=job-%j.err\nset -eo pipefail\ncd "$SLURM_SUBMIT_DIR"\nPYTHON_EXE="$1"\nMPI_MODULE="$2"\nif ! command -v module >/dev/null 2>&1 && [ -r /cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash ]; then\n  source /cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash\nfi\nexport M5S3_MODULE_STATUS="SKIPPED: module command unavailable; probing active compiler"\nif command -v module >/dev/null 2>&1; then\n  if [ -n "$MPI_MODULE" ]; then\n    if module load "$MPI_MODULE"; then export M5S3_MODULE_STATUS="LOADED: $MPI_MODULE";\n    else export M5S3_MODULE_STATUS="FAILED: $MPI_MODULE; probing active compiler"; fi\n  else export M5S3_MODULE_STATUS="UNCHANGED: active site environment"; fi\n  module list 2>&1 || true\nfi\nexport OMP_NUM_THREADS=2 OMP_DYNAMIC=FALSE OMP_PROC_BIND=close OMP_PLACES=cores\n# One task with two CPU slots; every step stays within this resource shape.\nsrun --exact --ntasks=1 --cpus-per-task=2 --cpu-bind=cores "$PYTHON_EXE" worker.py\n', 'container_examples.py': '"""Bounded discovery and real CPU examples using existing local course images."""\nimport json, math, os, platform, shutil, subprocess, sys\nfrom pathlib import Path\n\nCASE_NAMES=(\'container_image\',\'container_environment\',\'container_bind\',\'container\',\'container_numpy\')\n\ndef discover_images(explicit, roots, limit=12):\n    """Inspect configured course directories only; no recursive filesystem search."""\n    candidates=[]; diagnostics=[]; seen=set()\n    def add(path):\n        try:\n            p=Path(path).expanduser().resolve()\n            if p in seen or not p.is_file() or not os.access(p,os.R_OK):return\n            st=p.stat();seen.add(p)\n            candidates.append({\'path\':str(p),\'size_bytes\':st.st_size,\'mtime_ns\':st.st_mtime_ns})\n        except OSError as e:diagnostics.append(str(e))\n    if explicit:\n        add(explicit)\n        if not candidates:diagnostics.append(\'Configured IMAGE is missing or unreadable: \'+str(explicit))\n        return candidates,diagnostics\n    # Prefer the current course directory; the sponsor path is a historical fallback.\n    for folder in roots:\n        p=Path(folder).expanduser()\n        if not p.is_dir():diagnostics.append(\'Directory unavailable: \'+str(p));continue\n        try:\n            entries=[]\n            with os.scandir(p) as scan:\n                for i,entry in enumerate(scan):\n                    if i>=128:break\n                    if entry.name.lower().endswith((\'.sif\',\'.simg\')):entries.append(Path(entry.path))\n            entries.sort(key=lambda x:(not any(t in x.name.lower() for t in (\'python\',\'numpy\',\'pytorch\')),x.name))\n            for path in entries:\n                add(path)\n                if len(candidates)>=limit:return candidates,diagnostics\n        except OSError as e:diagnostics.append(str(e))\n    return candidates,diagnostics\n\ndef run_examples(cfg, root, report, record, sha, agree):\n    root=Path(root)\n    def remaining(status,reason):\n        for name in CASE_NAMES:\n            if name not in report[\'cases\']:record(name,status,reason=reason)\n    if not cfg.get(\'run_container\',True):\n        remaining(\'SKIPPED\',\'RUN_CONTAINER=False; deliberately disabled\');return\n    runtime=shutil.which(\'apptainer\') or shutil.which(\'singularity\')\n    if not runtime:\n        runtime=next((p for p in (\'/usr/bin/apptainer\',\'/usr/bin/singularity\') if os.access(p,os.X_OK)),None)\n    if not runtime:\n        remaining(\'SKIPPED\',\'No Apptainer/Singularity runtime visible inside the compute job\');return\n    candidates,diagnostics=discover_images(cfg.get(\'image\',\'\'),cfg[\'image_dirs\'])\n    report[\'container_discovery\']={\'runtime\':runtime,\'candidates\':candidates,\'diagnostics\':diagnostics}\n    if not candidates:\n        remaining(\'SKIPPED\',\'No readable image in configured course directories; inspect container_discovery or set IMAGE\');return\n    # Preserve the image environment but remove host bind/env overrides for this experiment.\n    env={k:v for k,v in os.environ.items() if not k.startswith((\'APPTAINERENV_\',\'SINGULARITYENV_\'))\n         and k not in (\'APPTAINER_BIND\',\'APPTAINER_BINDPATH\',\'SINGULARITY_BIND\',\'SINGULARITY_BINDPATH\')}\n    controls=\'OMP_NUM_THREADS=2,OPENBLAS_NUM_THREADS=2,MKL_NUM_THREADS=2,NUMEXPR_NUM_THREADS=2\'\n    prefix=[runtime,\'exec\',\'--cleanenv\',\'--containall\',\'--bind\',str(root)+\':/work\',\'--pwd\',\'/work\',\'--env\',controls]\n    def execute(args,timeout=25):\n        return subprocess.run([str(x) for x in args],capture_output=True,text=True,timeout=timeout,env=env,cwd=root)\n    def checked(args,timeout=25):\n        r=execute(args,timeout)\n        if r.returncode:raise RuntimeError(\'exit \'+str(r.returncode)+\': \'+(r.stderr+r.stdout)[-2000:])\n        return r.stdout.strip()\n    selected=None;probe=None;attempts=[]\n    for item in candidates[:4]:\n        image=Path(item[\'path\'])\n        for python in (\'python3\',\'python\'):\n            args=prefix+[image,python,\'environment_demo.py\']\n            try:\n                r=execute(args,8)\n                if r.returncode:raise RuntimeError((r.stderr+r.stdout)[-1200:])\n                value=json.loads(r.stdout)\n                version=tuple(int(x) for x in value[\'python\'].split(\'.\')[:2])\n                if version<(3,9):raise RuntimeError(\'Python 3.9+ is needed for the embedded payloads\')\n                selected=(image,python,args);probe=value;break\n            except (OSError,ValueError,KeyError,subprocess.TimeoutExpired,RuntimeError) as e:\n                attempts.append({\'image\':str(image),\'python\':python,\'diagnostic\':str(e)})\n        if selected:break\n    report[\'container_discovery\'][\'attempts\']=attempts\n    if not selected:\n        record(\'container_image\',\'FAIL\',reason=\'Readable images were found, but no attempted image launched a suitable Python; inspect diagnostics\')\n        remaining(\'SKIPPED\',\'No compatible image was selected\');return\n    image,python,probe_args=selected\n    try:\n        before=image.stat();image_hash=sha(image);after=image.stat()\n        if (before.st_size,before.st_mtime_ns)!=(after.st_size,after.st_mtime_ns):raise RuntimeError(\'Image changed during fingerprinting\')\n        if cfg.get(\'expected_image_sha256\') and image_hash!=cfg[\'expected_image_sha256\']:\n            raise RuntimeError(\'Image SHA-256 differs from EXPECTED_IMAGE_SHA256\')\n        report[\'container_image\']={\'path\':str(image),\'sha256\':image_hash,\'size_bytes\':after.st_size,\n                                   \'mtime_ns\':after.st_mtime_ns,\'python_command\':python,\n                                   \'runtime_version\':checked([runtime,\'--version\'])}\n        record(\'container_image\',\'PASS\',**report[\'container_image\'])\n    except Exception as e:\n        record(\'container_image\',\'FAIL\',reason=str(e));remaining(\'SKIPPED\',\'Image identity check failed\');return\n    def run_python(script,*args):return checked(prefix+[image,python,script,*args])\n    # Example 1: a real process inside the image, compared with the compute host.\n    try:\n        native=json.loads(checked([sys.executable,\'environment_demo.py\']))\n        ok=probe[\'kernel\']==native[\'kernel\'] and probe[\'machine\']==native[\'machine\']\n        record(\'container_environment\',\'PASS\' if ok else \'FAIL\',host=native,inside=probe,\n               command=[str(a) for a in probe_args],note=\'Python versions may differ; kernel and machine should agree on this host\')\n    except Exception as e:record(\'container_environment\',\'FAIL\',reason=str(e))\n    # Example 2: read the host CSV through /work and verify the written output on the host.\n    try:\n        target=root/\'written_by_container.json\';target.unlink(missing_ok=True)\n        printed=json.loads(run_python(\'bind_demo.py\'));written=json.loads(target.read_text())\n        ok=printed==written and written[\'input_sha256\']==sha(root/\'temperatures.csv\') and written[\'rows\']==cfg[\'reference\'][\'count\']\n        record(\'container_bind\',\'PASS\' if ok else \'FAIL\',result=written,host_output=str(target))\n    except Exception as e:record(\'container_bind\',\'FAIL\',reason=str(e))\n    # Example 3: unchanged scientific code, input and threshold in a different Python runtime.\n    try:\n        (root/\'container.json\').unlink(missing_ok=True)\n        run_python(\'analysis.py\',\'--input\',\'temperatures.csv\',\'--threshold\',cfg[\'threshold_c\'],\'--output\',\'container.json\')\n        got=json.loads((root/\'container.json\').read_text())\n        ref=json.loads((root/\'native.json\').read_text())[\'result\']\n        ok=agree(got[\'result\'],ref) and got[\'input_sha256\']==sha(root/\'temperatures.csv\') and got[\'code_sha256\']==sha(root/\'analysis.py\') and got[\'parameters\'][\'threshold_c\']==cfg[\'threshold_c\']\n        record(\'container\',\'PASS\' if ok else \'FAIL\',result=got[\'result\'],runtime=got[\'runtime\'],image_sha256=image_hash)\n    except Exception as e:record(\'container\',\'FAIL\',reason=str(e))\n    # Example 4: use NumPy already supplied by the image, if available; no installation.\n    try:\n        scale=cfg[\'load_scale\'];results=[]\n        for factor in (scale,2*scale):\n            r=execute(prefix+[image,python,\'numpy_demo.py\',str(factor)])\n            if r.returncode==77:\n                record(\'container_numpy\',\'SKIPPED\',reason=r.stdout.strip());break\n            if r.returncode:raise RuntimeError((r.stderr+r.stdout)[-2000:])\n            got=json.loads(r.stdout);d=got[\'displacements\'];f=[factor*x for x in (5.,6.,7.)]\n            if len(d)!=3 or not all(math.isfinite(v) for v in d):raise RuntimeError(\'Invalid displacement vector\')\n            k=((12.,-4.,2.),(-4.,8.,-2.),(2.,-2.,6.))\n            residual=max(abs(sum(a*b for a,b in zip(row,d))-force) for row,force in zip(k,f))\n            if got[\'forces\']!=f or residual>1e-10:raise RuntimeError(\'Independent stiffness-system residual check failed\')\n            results.append(got)\n        else:\n            ok=all(math.isclose(b,2*a,rel_tol=1e-12,abs_tol=1e-12) for a,b in zip(results[0][\'displacements\'],results[1][\'displacements\']))\n            record(\'container_numpy\',\'PASS\' if ok else \'FAIL\',results=results,note=\'Doubling force should double displacement; CPU only\')\n    except Exception as e:record(\'container_numpy\',\'FAIL\',reason=str(e))\n    final=image.stat()\n    if (after.st_size,after.st_mtime_ns)!=(final.st_size,final.st_mtime_ns):\n        record(\'container_image\',\'FAIL\',reason=\'Image changed while examples were running\')\n        record(\'container\',\'FAIL\',reason=\'Image changed while examples were running; cannot accept the recorded identity\')\n', 'environment_demo.py': "import json, os, platform, sys\nprint(json.dumps({'python':sys.version.split()[0], 'executable':sys.executable,\n                  'kernel':platform.release(), 'machine':platform.machine(),\n                  'omp_threads':os.environ.get('OMP_NUM_THREADS','unset')}))\n", 'bind_demo.py': "import csv, hashlib, json\nfrom pathlib import Path\npath=Path('temperatures.csv')\nwith path.open(newline='', encoding='utf-8') as f:\n    count=sum(1 for row in csv.DictReader(f))\nrecord={'rows':count, 'input_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),\n        'message':'The container read the saved host CSV and wrote this file.'}\nPath('written_by_container.json').write_text(json.dumps(record,indent=2)+'\\n')\nprint(json.dumps(record))\n", 'numpy_demo.py': '"""Small stiffness-system solve, on CPU only; no package installation."""\nimport importlib.util, json, sys\nif importlib.util.find_spec(\'numpy\') is None:\n    print(\'NumPy is not installed in this image.\'); raise SystemExit(77)\nimport numpy as np\nscale=float(sys.argv[1])\nk=np.array([[12.,-4.,2.],[-4.,8.,-2.],[2.,-2.,6.]])\nf=scale*np.array([5.,6.,7.])\nd=np.linalg.solve(k,f)\nprint(json.dumps({\'scale\':scale, \'forces\':f.tolist(), \'displacements\':d.tolist(),\n                  \'residual_inf\':float(np.max(np.abs(k@d-f))), \'numpy\':np.__version__}))\n'}
for name, source in SOURCES.items():
    (ROOT / name).write_text(source, encoding="utf-8")
print("Materialised:", ", ".join(SOURCES))


In [ ]:
rng = random.Random(23)
input_file = ROOT / "temperatures.csv"
with input_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, lineterminator="\n"); writer.writerow(["temperature_c"])
    writer.writerows([[f"{rng.uniform(10,30):.6f}"] for _ in range(1000)])
def host_run(threshold, output_name):
    output = ROOT / output_name
    print(call([sys.executable, ROOT/"analysis.py", "--input", input_file,
                "--threshold", threshold, "--output", output]))
    return json.loads(output.read_text())
first = host_run(THRESHOLD_C, "host_a.json")
second = host_run(THRESHOLD_C, "host_b.json")
changed = host_run(20.0 if THRESHOLD_C != 20.0 else 25.0, "host_changed.json")
assert first["result"] == second["result"]
assert first["input_sha256"] == changed["input_sha256"]
assert first["result"]["mean_c"] == changed["result"]["mean_c"]
assert first["result"]["fraction_above"] != changed["result"]["fraction_above"]
EVIDENCE["E3_recorded_python"] = "PASS"
print("LOCAL PASS: repeat agrees; a changed threshold changes the fraction only.")
print(json.dumps(first, indent=2))


In [ ]:
# Real computed summary, not an illustrative performance chart.
from html import escape
rows = [("Baseline", THRESHOLD_C, first["result"]),
        ("Changed", changed["parameters"]["threshold_c"], changed["result"])]
print("Run       threshold C    mean C    fraction above")
for name, threshold, r in rows:
    print(f"{name:9} {threshold:11.1f} {r['mean_c']:9.5f} {r['fraction_above']:15.3f}")
svg = ['<svg xmlns="http://www.w3.org/2000/svg" width="720" height="180" viewBox="0 0 720 180">',
       '<rect width="720" height="180" fill="white"/>',
       '<text x="20" y="25" font-family="sans-serif" font-size="18">Computed fraction above threshold (0 to 1)</text>']
for i,(name,threshold,r) in enumerate(rows):
    y = 50+i*55
    svg += [f'<text x="20" y="{y+20}" font-family="sans-serif" font-size="16">{escape(name)} {threshold:g} C</text>',
            f'<rect x="210" y="{y}" width="{450*r["fraction_above"]:.2f}" height="28" fill="{["#356487","#6A8A76"][i]}"/>',
            f'<text x="665" y="{y+20}" font-family="sans-serif" font-size="15">{r["fraction_above"]:.3f}</text>']
svg.append('</svg>'); svg = "\n".join(svg)
(ROOT/"fractions.svg").write_text(svg)
try:
    from IPython.display import SVG, display
    display(SVG(data=svg))
except ImportError:
    print("Plot saved to", ROOT/"fractions.svg", "(IPython display unavailable here)")


### E3a · Inspect the actual C/OpenMP loop
The same CSV, threshold and units feed serial C and two OpenMP threads. Each thread keeps private partial sums, combined by a reduction. Floating-point addition order can differ from Python `math.fsum`; compare with a declared teaching tolerance, not exact JSON bytes.

```c
double sum = 0;
int above = 0;
#pragma omp parallel num_threads(2) reduction(+:sum,above)
{
    #pragma omp for schedule(static)
    for (int i = 0; i < n; i++) {
        sum += values[i];
        above += values[i] > threshold;
    }
}
```
This is a readable excerpt of `temperature.c`; the full program records the actual thread count. It is not a new performance benchmark. The input is deliberately tiny.

**Student experiment:** change only `THRESHOLD_C` to 22, rerun E3 and the preparation/submission/collection below. Predict unchanged mean and input identity, and a changed fraction. Avoid changing image/toolchain at the same time.

**Takeaway:** code + data + parameters + runtime identity make a scientific result explainable.


### Checkpoint Q3
The same input file is analysed at thresholds 25 C and 20 C. Which comparison is expected?

- **A.** The mean must change because the threshold changed
- **B.** The input hash changes automatically
- **C.** Only the fraction above threshold and recorded parameter need change
- **D.** Every byte of the full provenance JSON must match


**Checkpoint Q3 · Answer: C**

The mean is computed from the same input values and should not change with the threshold. The fraction above threshold and the recorded threshold parameter can change. The input hash remains the same.


## E4 · Launch existing cluster containers: four concrete examples
**Real problem:** run useful software already packaged at the cluster, expose saved host data, and verify results rather than stopping at an image filename.

**Prediction:** can host and container Python versions differ while their Linux kernel agrees? Will a file written under `/work` appear in the host run directory? Should the saved temperature result change?

The same bounded CPU batch now launches:
1. **Environment:** execute `environment_demo.py` inside the selected image. Compare its Python/executable with the compute host and check the shared kernel/machine.
2. **Bind and output:** read the saved CSV at `/work/temperatures.csv`, write `written_by_container.json`, and verify that output and input identity on the host.
3. **Scientific replay:** execute the unchanged `analysis.py` in the image and compare input/code hashes, threshold, count, mean and fraction with the host result.
4. **NumPy already in the image:** solve a tiny stiffness system `K d = F`, check the residual independently, double the force and check that displacement doubles. If NumPy is absent this additional example is SKIPPED; the first three still run. No GPU or package installation is needed.

For each payload the worker uses real commands of this form (the selected image/Python are printed in the report):
```bash
apptainer exec --cleanenv --containall \
  --bind "$RUN_DIR:/work" --pwd /work "$IMAGE" \
  python3 environment_demo.py
```
The worker explicitly caps OpenMP/BLAS threads at two and accepts `python` if that is the image's executable. The code below displays files before submission. **These commands execute in the Slurm job, not merely in this explanation.**


### E4a · Find the existing images and inspect the payloads
This inventory only lists configured directories. The compute job performs the actual compatibility probes and launches. Current course path: `/data/software/containers`; the older sponsor path is only a fallback. Add a directory only when your instructor confirms it belongs to the course.


In [ ]:
import runpy
helpers = runpy.run_path(str(ROOT/"container_examples.py"))
IMAGE_CANDIDATES, IMAGE_DIAGNOSTICS = helpers["discover_images"](IMAGE, IMAGE_DIRS)
print("NOTEBOOK-HOST INVENTORY (compute-node visibility is checked in the job)")
for item in IMAGE_CANDIDATES:
    print(f"{item['size_bytes']/1024**3:.2f} GiB  {item['path']}")
for message in IMAGE_DIAGNOSTICS: print(message)
if not IMAGE_CANDIDATES:
    print("No image visible here. Inspect IMAGE_DIRS or set IMAGE; the job also checks its own host.")
for name in ("environment_demo.py","bind_demo.py","numpy_demo.py"):
    print("\n---",name,"---\n"+(ROOT/name).read_text())


In [ ]:
assert math.isfinite(THRESHOLD_C) and -100 <= THRESHOLD_C <= 100
assert math.isfinite(LOAD_SCALE) and 0.1 <= LOAD_SCALE <= 10
if EXPECTED_IMAGE_SHA256:
    assert len(EXPECTED_IMAGE_SHA256)==64 and all(c in "0123456789abcdef" for c in EXPECTED_IMAGE_SHA256.lower())
image = Path(IMAGE).expanduser().resolve() if IMAGE else None
image_hash = None  # filled only from the completed compute job's selected-image evidence
frozen = {name: (ROOT/name).read_text() for name in SOURCES}
frozen["temperatures.csv"] = input_file.read_text()
identity = {"source_sha256": {name: hashlib.sha256(text.encode()).hexdigest() for name,text in frozen.items()},
            "threshold_c":THRESHOLD_C, "run_container":RUN_CONTAINER,
            "image":str(image) if image else "", "image_dirs":list(IMAGE_DIRS),
            "expected_image_sha256":EXPECTED_IMAGE_SHA256.lower(), "load_scale":LOAD_SCALE,
            "module":MODULE, "python_executable":sys.executable, "reference":first["result"]}
fingerprint = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:20]
RUN_DIR = ROOT / ("run-"+fingerprint+("-"+uuid.uuid4().hex[:6] if NEW_RUN else ""))
RUN_DIR.mkdir(exist_ok=True)
for name, text in frozen.items():
    target = RUN_DIR/name
    if target.exists() and target.read_text() != text: raise RuntimeError("Frozen content mismatch: "+name)
    if not target.exists(): target.write_text(text, encoding="utf-8")
settings = dict(identity, run_id=RUN_DIR.name)
settings_path = RUN_DIR/"settings.json"
if settings_path.exists(): assert json.loads(settings_path.read_text()) == settings
else: settings_path.write_text(json.dumps(settings,indent=2))
print("Frozen run:", RUN_DIR.name)
print("RUN_CONTAINER:", RUN_CONTAINER, "IMAGE:", str(image) if image else "AUTO: existing course images")
print("Cases: NOT RUN until this job's completed report is collected.")
print((RUN_DIR/"run.sbatch").read_text())


In [ ]:
JOB_ID = None
state_file = RUN_DIR/"job_id.txt"
if state_file.exists():
    JOB_ID = state_file.read_text().strip()
    if not JOB_ID.isdigit(): raise RuntimeError("Invalid saved job ID")
    print("Reusing job", JOB_ID, "; no duplicate submission.")
elif not RUN_CLUSTER:
    print("NOT RUN: RUN_CLUSTER=False.")
elif not shutil.which("sbatch"):
    print("NOT RUN: sbatch unavailable; local Python evidence is separate.")
else:
    # A different settings fingerprint must not create another job while one is active.
    for previous in ROOT.glob("run-*/job_id.txt"):
        if previous.parent == RUN_DIR: continue
        previous_id = previous.read_text().strip()
        previous_log = previous.parent/("job-"+previous_id+".out")
        if previous_log.exists() and "M5S3_JOB_DONE status=" in previous_log.read_text(errors="replace"): continue
        if not shutil.which("squeue"):
            raise RuntimeError("NOT RUN: verify previous job "+previous_id+" before another submission")
        q = subprocess.run(["squeue","-h","-j",previous_id],capture_output=True,text=True,timeout=10)
        if q.returncode or q.stdout.strip():
            raise RuntimeError("NOT RUN: earlier job is active or unverified: "+previous_id)
    # Avoid inheriting Jupyter's Slurm task/memory settings into a new allocation.
    submit_env = {k:v for k,v in os.environ.items() if not k.startswith("SLURM_")}
    result = subprocess.run(["sbatch", "--parsable", "run.sbatch", sys.executable, MODULE],
                            cwd=RUN_DIR, env=submit_env, capture_output=True, text=True, timeout=25)
    if result.returncode:
        print("FAIL: submission rejected:", result.stderr or result.stdout)
    else:
        JOB_ID = result.stdout.strip().split(";")[0]
        if not JOB_ID.isdigit(): raise RuntimeError("Unexpected sbatch response: "+result.stdout)
        state_file.write_text(JOB_ID)
        print("Submitted CPU job", JOB_ID)
print("Do not resubmit while queued. Rerun the COLLECTION cell below.")


In [ ]:
BATCH_REPORT = None
if not JOB_ID:
    print("NOT RUN: no batch job for this run.")
else:
    report_path = RUN_DIR/"report.json"
    log_path = RUN_DIR/("job-"+JOB_ID+".out")
    err_path = RUN_DIR/("job-"+JOB_ID+".err")
    log = log_path.read_text(errors="replace") if log_path.exists() else ""
    if report_path.exists() and "M5S3_JOB_DONE status=" in log:
        candidate = json.loads(report_path.read_text())
        if not candidate.get("complete") or candidate.get("run_id") != RUN_DIR.name or str(candidate.get("job_id")) != JOB_ID:
            raise RuntimeError("FAIL: report/job identity mismatch")
        BATCH_REPORT = candidate
        image_hash = candidate.get("container_image",{}).get("sha256")
        print("REAL BATCH REPORT", json.dumps(candidate,indent=2))
        print("\nCASE SUMMARY")
        for name, case in candidate["cases"].items():
            print(f"{name:25} {case['status']:8} {case.get('reason','')}")
        if candidate.get("container_image"):
            print("Selected image:",candidate["container_image"]["path"])
            print("SHA-256:",image_hash)
            print("To reuse exactly this image, set IMAGE and EXPECTED_IMAGE_SHA256 from this report.")
    else:
        states = []
        for command in (["squeue","-h","-j",JOB_ID,"-o","%T|%R"],
                        ["sacct","-n","-P","-X","-j",JOB_ID,"-o","State,ExitCode"]):
            if shutil.which(command[0]):
                try:
                    q = subprocess.run(command,capture_output=True,text=True,timeout=10)
                    if q.returncode == 0 and q.stdout.strip(): states.append(q.stdout.strip())
                except (OSError,subprocess.TimeoutExpired): pass
        details = " | ".join(states)
        terminal = any(token in details for token in ("FAILED","CANCELLED","TIMEOUT","OUT_OF_MEMORY","NODE_FAIL","COMPLETED"))
        print(("FAIL: terminal job has no complete evidence. " if terminal else "QUEUED/RUNNING/UNKNOWN: no completed evidence yet. ")+details)
        print("Inspect: squeue -j", JOB_ID, "or sacct -j", JOB_ID)
    if err_path.exists() and err_path.stat().st_size:
        print("Actual stderr:", err_path.read_text(errors="replace")[-2500:])
    if not BATCH_REPORT:
        print("Log tail:", log[-1500:])
        print("Rerun this collection cell later; do not submit again.")


**Result / Explanation:** use the actual `CASE SUMMARY`, selected-image identity and printed results. Native checks and each container example have separate statuses. A job-level PASS does not promote a SKIPPED capability to PASS.

- `container_image`: a candidate actually launched a suitable Python and its image identity was recorded.
- `container_environment`: host and container share the kernel/machine; Python versions and paths can differ. Equal Python version strings would not invalidate the demonstration.
- `container_bind`: the process read the saved host CSV and its written JSON was verified from the host.
- `container`: the unchanged temperature calculation agrees numerically and identifies the same code/input bytes.
- `container_numpy`: the already packaged NumPy solved the small system; the host independently checked residuals and load scaling. This is a correctness example, not a benchmark.

**Student experiment:** change only `LOAD_SCALE` to 0.5, predict half the original displacement, and prepare/submit a new run **after the prior job finishes**. Alternatively change only `THRESHOLD_C` to 22 and predict the effect on the temperature result. Do not change image and parameter together.

**Troubleshooting:** inspect `container_discovery` for every attempted image and executable. An image might be readable on Jupyter but absent on a compute node, lack Python, or require unsupported runtime features. Set `IMAGE` or edit `IMAGE_DIRS` to the actual instructor-approved location; do not download/build another image to hide the failure. After an accepted AUTO run, set the explicit image path and `EXPECTED_IMAGE_SHA256` for a pinned rerun.

**Takeaway:** existing images become useful when the payload, mount, resources and correctness checks are explicit.

### Optional MPI/GPU boundaries (no computation)
A container CPU success does not validate host/container MPI, PMI/PMIx, interconnects or GPU drivers. Use only a site-approved launcher. No MPI/GPU job or `--nv` is needed for these examples.


### Checkpoint Q4
The SIF exists, but the analysis cannot open /work/temperatures.csv. Which check is most useful?

- **A.** Increase the number of GPUs
- **B.** Check the bind source/destination and working directory
- **C.** Rename the image to match the CSV
- **D.** Request more MPI ranks


**Checkpoint Q4 · Answer: B**

Check how the host directory is bound into the container and where the program is running. The existence of a SIF image alone does not make the CSV visible at /work/temperatures.csv.


### Checkpoint Q5
The container ran successfully on one CPU node. What follows about a later MPI/GPU run?

- **A.** The image automatically reserves a GPU
- **B.** MPI and GPU compatibility still require their own checks and allocations
- **C.** Two-node MPI is guaranteed to work
- **D.** The runtime will be identical on every system


**Checkpoint Q5 · Answer: B**

A successful CPU container run does not demonstrate MPI or GPU operation. Those require separate compatibility checks, hardware allocations and tests. It does not guarantee the same execution time.


## E5 · Cache correctly and hand over a replayable experiment
**Real problem:** a cached answer can become stale when parameters, code or data change.

**Prediction:** which calls should compute: threshold 25, threshold 25 again, threshold 20?

**Code / Run:** a small teaching cache includes input bytes, code bytes, parameters and runtime identity. This is not a run of Snakemake or Nextflow. Real workflow tools coordinate task dependencies and execution environments; we isolate one correctness rule here.


In [ ]:
cache_file = ROOT/"cache_index.json"
def cached_run(threshold):
    key_data = {"input":sha(input_file), "code":sha(ROOT/"analysis.py"),
                "threshold_c":threshold, "python":sys.version, "executable":sys.executable,
                "system":platform.system(), "machine":platform.machine()}
    key = hashlib.sha256(json.dumps(key_data,sort_keys=True).encode()).hexdigest()
    index = json.loads(cache_file.read_text()) if cache_file.exists() else {}
    output = ROOT/("cached_"+key+".json")
    if key in index and output.exists() and sha(output) == index[key]:
        return "REUSE", json.loads(output.read_text())
    r = host_run(threshold,output.name)
    index[key] = sha(output); cache_file.write_text(json.dumps(index,indent=2))
    return "COMPUTE", r
cache_file.write_text("{}")  # reset only this teaching cache index
cache_results = [cached_run(t) for t in (25,25,20)]
print("Observed actions:", [x[0] for x in cache_results])
assert [x[0] for x in cache_results] == ["COMPUTE","REUSE","COMPUTE"]
EVIDENCE["E5_cache"] = "PASS"
print("LOCAL PASS: parameter change invalidates stale output.")


### E5a · Independent replay and deliberate integrity failure
A second process should reproduce the result without your notebook variables. The handover contains code, input, parameters, an expected result and file identities. A replay first checks file hashes, then numerical agreement. A separate damaged copy must be rejected before scientific calculation.

**Student experiment:** change only threshold 22 in `cached_run(22)`, predict a cache miss, then rerun it unchanged and predict reuse. For integrity, change one input row in the damaged copy only.

**Takeaway:** a reusable environment still needs an explicit workflow, inputs and validation rule.


In [ ]:
handover = ROOT/("handover-"+uuid.uuid4().hex[:8])
handover.mkdir()
for name in ("analysis.py","replay.py","experiment.def","spack.yaml"):
    shutil.copy2(ROOT/name,handover/name)
shutil.copy2(input_file,handover/"temperatures.csv")
# A fresh host record for the selected parameter, independent of cached output.
final = host_run(THRESHOLD_C,"handover_expected.json")
shutil.copy2(ROOT/"handover_expected.json",handover/"expected.json")
manifest = {"threshold_c":THRESHOLD_C,
            "files":{p.name:sha(p) for p in handover.iterdir() if p.is_file()},
            "runtime":final["runtime"], "image_sha256":image_hash if RUN_CONTAINER else None,
            "image_path":BATCH_REPORT.get("container_image",{}).get("path") if BATCH_REPORT else None,
            "container_status":BATCH_REPORT["cases"].get("container",{}).get("status","NOT RUN") if BATCH_REPORT else "NOT RUN",
            "comparison":{"rel_tol":1e-12,"abs_tol":1e-12,"note":"Teaching tolerance"}}
(handover/"manifest.json").write_text(json.dumps(manifest,indent=2))
(handover/"README.txt").write_text("Use Python 3.9+ and run: python3 replay.py\nCore replay uses stdlib; venv is recreated, not copied.\nThe definition uses a mutable teaching tag; archive a trusted SIF and its identity for container replay.\nNo MPI/GPU run is included.\n")
print(call([sys.executable,handover/"replay.py"],cwd=handover))
EVIDENCE["E5_independent_replay"] = "PASS"
damaged = ROOT/(handover.name+"-damaged")
shutil.copytree(handover,damaged)
with (damaged/"temperatures.csv").open("a") as f: f.write("99.000000\n")
rejected = subprocess.run([sys.executable,damaged/"replay.py"],cwd=damaged,capture_output=True,text=True,timeout=20)
assert rejected.returncode == 2 and "integrity mismatch" in rejected.stdout
print("LOCAL PASS: deliberate tampering rejected:",rejected.stdout.strip())
EVIDENCE["E5_tamper_rejection"] = "PASS"
archive = ROOT/(handover.name+".zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in handover.iterdir():
        if p.is_file():z.write(p,p.name)
print("Shareable handover:",archive)


### Checkpoint Q6
A workflow’s input filename is unchanged, but the CSV contents or threshold changed. What should happen?

- **A.** Reuse the old result because the filename matches
- **B.** Recompute using input identity and parameters in the cache key
- **C.** Change only the output label
- **D.** Remove provenance so the outputs look identical


**Checkpoint Q6 · Answer: B**

The cache must account for the actual input contents and analysis parameters. If either changes, the previous result may be stale and the calculation must be repeated.


## Final experiment · Replay the PyTorch container laboratory in Jupyter

The SSH practical trained a small model in three separate container processes. The Jupyter cell below repeats **the same program and the same parameters** without opening SSH or copying any files. It performs two identical seeded runs and a third run with a changed target slope, then checks both reproducibility and parameter sensitivity.

It prints the learned slope, intercept, error and fingerprint for runs A, B and C, followed by `PASS` or `FAIL` for each checkpoint. It runs on CPU, needs no bind mount, and does not install anything. If this Jupyter environment cannot access the shared image or Apptainer, it reports `NOT RUN` rather than claiming a successful container experiment.

Compare the results with `~/hpc_course/m5s3_container_lab/run_A.txt`, `run_B.txt`, and `run_C.txt` from your SSH session. Pay attention to how the same image can support either reproducible results or deliberately different results when input parameters change.


In [ ]:
# Automatically repeat the introductory PyTorch training experiment.
# Three separate Apptainer executions: same parameters twice, then a changed slope.
import shutil as _shutil, subprocess as _subprocess, json as _json
from pathlib import Path as _Path

_training_program = "import sys, json, hashlib\nimport torch\n\nseed = int(sys.argv[1])\ntrue_slope = float(sys.argv[2])\ntorch.set_num_threads(2)\ntorch.manual_seed(seed)\n\nx = torch.linspace(-2, 2, 256).reshape(-1, 1)\ny = true_slope * x + 2.0 + 0.05 * torch.sin(x * 9)\nmodel = torch.nn.Linear(1, 1)\noptimizer = torch.optim.SGD(model.parameters(), lr=0.05)\n\nfor epoch in range(240):\n    optimizer.zero_grad()\n    loss = torch.mean((model(x) - y) ** 2)\n    loss.backward()\n    optimizer.step()\n\nwith torch.no_grad():\n    learned_slope = float(model.weight.item())\n    learned_intercept = float(model.bias.item())\n    error = float(torch.mean(torch.abs(model(x) - y)).item())\n\nresult = {\n    \"torch\": torch.__version__,\n    \"seed\": seed,\n    \"target_slope\": true_slope,\n    \"learned_slope\": round(learned_slope, 6),\n    \"learned_intercept\": round(learned_intercept, 6),\n    \"mean_absolute_error\": round(error, 6),\n}\nresult[\"fingerprint\"] = hashlib.sha256(\n    json.dumps(result, sort_keys=True).encode()\n).hexdigest()[:12]\nprint(\"M5S3_TRAIN=\" + json.dumps(result, sort_keys=True))\n"
_training_image = _Path("/data/software/containers/pytorch-2.8.0-cuda12.8.sif")
_apptainer_bin = _shutil.which("apptainer")
_training_runs = {}
if not _apptainer_bin:
    print("NOT RUN: Apptainer is not available to the Jupyter kernel.")
elif not _training_image.is_file():
    print("NOT RUN: the shared PyTorch image is not visible:", _training_image)
else:
    for _name, _slope in [("A", "3.0"), ("B", "3.0"), ("C", "4.0")]:
        _command = [_apptainer_bin, "exec", "--cleanenv", str(_training_image),
                    "/opt/conda/bin/python3", "-", "42", _slope]
        try:
            _p = _subprocess.run(_command, input=_training_program, capture_output=True,
                                 text=True, timeout=120)
            if _p.returncode:
                print("FAIL run", _name, "return code", _p.returncode)
                print((_p.stderr or _p.stdout)[-1000:])
                break
            _markers = [line.split("M5S3_TRAIN=",1)[1] for line in _p.stdout.splitlines()
                        if line.startswith("M5S3_TRAIN=")]
            if not _markers:
                print("FAIL run", _name, ": no training result found")
                print(_p.stdout[-1000:])
                break
            _training_runs[_name] = _json.loads(_markers[-1])
            _r = _training_runs[_name]
            print("Run", _name, "| target slope", _r["target_slope"],
                  "| learned", _r["learned_slope"],
                  "| intercept", _r["learned_intercept"],
                  "| MAE", _r["mean_absolute_error"],
                  "| fingerprint", _r["fingerprint"])
        except (_subprocess.TimeoutExpired, OSError, ValueError) as _exc:
            print("FAIL run", _name, _exc)
            break
    if len(_training_runs) == 3:
        _same = _training_runs["A"] == _training_runs["B"]
        _changed = (_training_runs["A"]["fingerprint"] != _training_runs["C"]["fingerprint"]
                    and abs(_training_runs["A"]["learned_slope"] - 3) < 0.2
                    and abs(_training_runs["C"]["learned_slope"] - 4) < 0.2)
        print("CHECKPOINT 1 (same image, seed and inputs):", "PASS" if _same else "FAIL")
        print("CHECKPOINT 2 (changed target slope):", "PASS" if _changed else "FAIL")
        print("FINAL CONTAINER LAB:", "PASS" if (_same and _changed) else "FAIL")


## 6 · Four essentials and exit question
1. **Environment:** identify the interpreter, dependencies and compiler that actually ran.
2. **Container:** preserve user-space software while checking host kernel, hardware and runtime integration.
3. **Experiment:** record code, inputs, parameters and a justified numerical comparison.
4. **Workflow:** recompute when dependencies change and hand over a runnable replay.

**Exit question:** a teammate has your SIF but cannot reproduce the result. Name two missing experiment artifacts and one host dependency to check.

**Next session:** Session 24 is the group project workshop. Bring the handover ZIP, source/settings, validation status and one unresolved environment issue. No new theory reading is required for that workshop.


In [ ]:
print("LOCAL PYTHON EVIDENCE (this notebook execution):")
for name, status in EVIDENCE.items(): print(f"{name:30} {status}")
print("SEPARATE BATCH EVIDENCE:")
for name in ("python","c_serial","c_openmp","container_image","container_environment","container_bind","container","container_numpy"):
    case = BATCH_REPORT["cases"].get(name,{"status":"NOT RUN"}) if BATCH_REPORT else {"status":"NOT RUN"}
    print(f"{name:30} {case['status']}",case.get("reason",""))
print("Spack concretization/install: NOT RUN. MPI/GPU compute: NOT RUN.")
print("Local PASS does not prove SciTech, container or MPI/GPU PASS.")


## References and validation boundary
**Primary teaching reference:** official 2026–27 syllabus, Session 23, page 16. Main reading: Oscar Diez, *High Performance Computing: Applications and Frontiers*, Chapter 1, modern programming/reproducible science toolchains; supporting reading: Robey and Zamora, *Parallel and High Performance Computing*, Chapter 17. Builds on M3S4’s provenance checks and M5S2’s rank/thread application, then leads into the project workshop.

Official documentation checked 24 October 2026:
- [Modules](https://modules.readthedocs.io/en/stable/module.html)
- [Python venv](https://docs.python.org/3/library/venv.html)
- [Spack environments](https://spack.readthedocs.io/en/latest/environments.html)
- [Apptainer exec](https://apptainer.org/docs/user/main/cli/apptainer_exec.html), [binds](https://apptainer.org/docs/user/main/bind_paths_and_mounts.html), [MPI](https://apptainer.org/docs/user/main/mpi.html), [GPU support](https://apptainer.org/docs/user/main/gpu.html)
- [Slurm sbatch](https://slurm.schedmd.com/sbatch.html)
- [Snakemake](https://snakemake.readthedocs.io/), [Nextflow](https://www.nextflow.io/docs/latest/)

See README and VALIDATION.md for author validation. The published notebook has no prefilled fake SciTech outputs. Local author checks are documented separately; The instructor-provided export confirms the earlier native batch job 24608 on julia: Python, serial C and actual two-thread OpenMP PASS. The updated image-discovery/container examples still require their own live SciTech report; no new container PASS is prefilled.
